# figureS10\n\nRun this notebook to regenerate the corresponding SVG from `../results_processed/figure_svg_payloads.csv`.\n

In [ ]:
from pathlib import Path
import base64
import csv
import gzip
import hashlib
import sys

csv.field_size_limit(sys.maxsize)

FIGURE_ID = 'figureS10'

def find_repo_root():
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / "results_processed" / "figure_svg_payloads.csv").exists():
            return path
    raise FileNotFoundError("Could not find results_processed/figure_svg_payloads.csv from the current working directory")

repo_root = find_repo_root()
processed_csv = repo_root / "results_processed" / "figure_svg_payloads.csv"
output_svg = repo_root / "figures" / f"{FIGURE_ID}.svg"

with processed_csv.open(newline="") as handle:
    row = next(row for row in csv.DictReader(handle) if row["figure_id"] == FIGURE_ID)

svg_bytes = gzip.decompress(base64.b64decode(row["svg_gzip_base64"]))
actual_sha256 = hashlib.sha256(svg_bytes).hexdigest()
if actual_sha256 != row["source_svg_sha256"]:
    raise ValueError(f"Checksum mismatch for {FIGURE_ID}: {actual_sha256}")
output_svg.write_bytes(svg_bytes)
